# JIN — Kaggle GPU, prepared_v4
Upload bằng scripts/pack_kaggle.py; giữ Dataset/Notebook PRIVATE. Settings → GPU, Internet nếu cần install. Ô 1 tự dò upload_manifest.json + parent checkpoint trong /kaggle/input; chỉ điền tay khi bị dừng. Ô 1: MODE='instruct' tiếp model chat hiện có, KHÔNG random-init; MODE='pretrain' khi gói pin checkpoint có optimizer.
Đang pin cho run general_v1 continued-pretrain (MODE='pretrain', RUN_LONG=True, ALLOW_LONG=True, STEPS=50.000): ô 4 chạy smoke 20 bước trước, ô 5 chạy dài tiếp trong cùng một version. Muốn chỉ smoke thì đặt RUN_LONG=False; chỉnh STEPS theo it/s đo được ở ô 4 (bảng ước tính trong KAGGLE_V4_VI.md §5). Không hứa tốc độ/chất lượng chưa đo. Output phải được lưu trước khi hết session/quota.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
# Ô này TỰ DÒ upload_manifest.json + parent checkpoint trong /kaggle/input. Copy file path trong panel Input là path của FILE, không phải thư mục.
INPUT = Path('/kaggle/input')
DATA_INPUT = None            # chỉ điền tay khi tự dò báo lỗi, ví dụ Path('/kaggle/input/datasets/<owner>/jin-v4-training')
CHECKPOINT_INPUT = None      # chỉ điền tay khi tự dò báo lỗi, ví dụ Path('/kaggle/input/jin-ckpt/jingpt_chat.pt')
EXTRACTED_ROOTS = {}         # chỉ điền tay khi Kaggle giải nén ra thư mục con, ví dụ {'jin-code.zip': Path('/kaggle/input/.../jin-code')}
MODE = 'pretrain'  # đêm nay: continued pretrain general_v1 từ checkpoint v4 (có optimizer).
RUN_LONG = True    # False = chỉ smoke 20 bước; True = chạy dài trong version này.
ALLOW_LONG = True  # bắt buộc khi STEPS > 5.000 (ô 5 chặn 5.000 nếu False).
STEPS = 50000      # ~7h15 ở 1,914 it/s đo từ run v4; chỉnh theo it/s in ở ô 4.
ROOT = Path('/kaggle/working/jin')
def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(4 << 20), b''): h.update(block)
    return h.hexdigest()
assert MODE in ('instruct', 'pretrain')
if DATA_INPUT is None:
    manifests = [p for p in INPUT.rglob('upload_manifest.json') if p.is_file()]
    assert len(manifests) == 1, f'Cần đúng 1 upload_manifest.json trong /kaggle/input; thấy: {manifests}'
    DATA_INPUT = manifests[0].parent
assert (DATA_INPUT / 'upload_manifest.json').is_file(), f'Thiếu upload_manifest.json trong {DATA_INPUT}'
bundle = json.loads((DATA_INPUT / 'upload_manifest.json').read_text())
if CHECKPOINT_INPUT is None:
    parent_name = bundle['parent']['filename']
    seen = [p for p in INPUT.rglob(parent_name) if p.is_file()]
    found = [p for p in seen if sha(p) == bundle['parent']['sha256']]
    assert found, f'Không thấy {parent_name} đúng sha trong /kaggle/input (đã xét {seen}); attach dataset chứa parent checkpoint'
    CHECKPOINT_INPUT = found[0]
assert CHECKPOINT_INPUT.is_file(), 'Thiếu parent checkpoint: sửa CHECKPOINT_INPUT'
assert sha(CHECKPOINT_INPUT) == bundle['parent']['sha256'], 'Sai parent checkpoint; không bỏ qua hash'
print('DATA_INPUT:', DATA_INPUT, 'CHECKPOINT_INPUT:', CHECKPOINT_INPUT)
assert not ROOT.exists(), 'Workspace đã có: restart session hoặc chọn workspace mới, không copy đè'
ROOT.mkdir(parents=True)
def extracted_root(filename, allowed):
    if filename in EXTRACTED_ROOTS: return EXTRACTED_ROOTS[filename]
    if all((DATA_INPUT / name).is_file() for name in allowed): return DATA_INPUT
    for rel in sorted(allowed, key=lambda n: (n.count('/'), n)):
        for p in INPUT.rglob(rel.rsplit('/', 1)[-1]):
            if p.is_file() and p.as_posix().endswith('/' + rel) and sha(p) == allowed[rel]['sha256']:
                return p.parents[rel.count('/')]
    return None
for item in bundle['archives']:
    archive_path = DATA_INPUT / item['filename']
    allowed = {f['path']: f for f in item['files']}
    for name in allowed:
        assert (ROOT / name).resolve().is_relative_to(ROOT) and chr(92) not in name, 'Unsafe path'
    if archive_path.is_file():
        assert sha(archive_path) == item['sha256'], f'Sai archive hash: {archive_path}'
        with zipfile.ZipFile(archive_path) as z:
            assert set(z.namelist()) == set(allowed), 'Archive entries khác manifest'
            z.extractall(ROOT)
    else:
        source_root = extracted_root(item['filename'], allowed)
        assert source_root, 'Không thấy bản đã giải nén; điền EXTRACTED_ROOTS: ' + item['filename']
        print('Extracted input reuse:', item['filename'], '->', source_root)
        for name, info in allowed.items():
            source = source_root / name
            assert source.is_file() and not source.is_symlink(), f'Thiếu file đã giải nén: {source}'
            assert sha(source) == info['sha256'], f'Sai input hash: {source}'
            destination = ROOT / name
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
    for name, info in allowed.items():
        assert sha(ROOT / name) == info['sha256'], f'Sai extracted hash: {name}'
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
from storage import configure_runtime
configure_runtime(ROOT)
os.environ['JINGPT_DEVICE'] = 'cuda'
os.environ['JINGPT_COMPILE'] = '0'
for name in ('JINGPT_TOKEN_CACHE', 'JINGPT_ALLOW_UNVERIFIED_CACHE'): os.environ.pop(name, None)
print('Workspace:', ROOT, '\nFree GiB:', round(shutil.disk_usage(ROOT).free / 2**30, 2))


In [ ]:
# Không reinstall torch CUDA khi Kaggle đã có. Chỉ install gói THIẾU; cần Internet nếu thiếu.
import importlib.util
missing = [name for name in ('numpy', 'flask') if importlib.util.find_spec(name) is None]
if missing: subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import torch
assert torch.cuda.is_available(), 'Bật GPU trong Settings, không train CPU âm thầm'
print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda)
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(i, p.name, round(p.total_memory / 2**30, 2), 'GiB')
print('Trainer hiện dùng cuda:0; không tự dùng nhiều GPU.')


In [ ]:
# Tái pin mixture Windows sang Kaggle SAU khi xác minh archive/data. Không patch source code.
from lineage import atomic_json, object_hash
from data_pipeline import verify_prepared
PREPARED = ROOT / bundle['prepared_relative']
m = json.loads((PREPARED / 'mixture.json').read_text())
r = json.loads((PREPARED / 'manifest.json').read_text())
assert object_hash(m) == r['mixture_hash'] and r['status'] == 'ready'
m['domains'] = {d: str(PREPARED / 'pretrain' / (d + '.txt')) for d in m['domains']}
m['validation_path'] = str(PREPARED / 'pretrain/validation.txt')
m['provenance_report'] = str(PREPARED / 'manifest.json')
r['relocated_from_mixture_hash'] = r['mixture_hash']
r['mixture_hash'] = object_hash(m)
atomic_json(str(PREPARED / 'mixture.json'), m)
atomic_json(str(PREPARED / 'manifest.json'), r)
print(verify_prepared(PREPARED))
from utils import load_checkpoint_file
from bpe_tokenizer import BPETokenizer
parent = load_checkpoint_file(str(CHECKPOINT_INPUT), map_location='cpu')
assert parent.get('tokenizer_state'), 'Parent thiếu tokenizer_state; không đoán tokenizer'
tok = BPETokenizer(); tok.load(str(ROOT / 'bpe.json'))
assert object_hash(parent['tokenizer_state']) == object_hash(tok.to_state_dict()), 'Tokenizer khác parent: đóng gói đúng tokenizer, không rebuild BPE'
PARENT_OPTIMIZER = None
if MODE == 'pretrain':
    opt = parent.get('optimizer_state_dict')
    assert opt and opt.get('param_groups'), 'Continued pretrain yêu cầu training checkpoint có optimizer state'
    groups = opt['param_groups']
    if any(g.get('mode') == 'muon' for g in groups): PARENT_OPTIMIZER = 'muon'
    elif all('betas' in g and g.get('mode', 'adamw') == 'adamw' for g in groups): PARENT_OPTIMIZER = 'adamw'
    else: raise ValueError('Optimizer không nhận dạng được; cần review trước khi resume')
print('Parent step:', parent.get('iteration'), 'Model config:', parent.get('model_config'))
del parent
import gc; gc.collect()


In [ ]:
def recipe(parent_path, output, steps, resume=False):
    inputs = {'tokenizer': {'path': 'bpe.json', 'sha256': sha(ROOT / 'bpe.json')}}
    training = {'mode': MODE, 'device': 'cuda', 'optimizer': PARENT_OPTIMIZER or 'adamw',
        'batch-size': 1, 'grad-accum': 4, 'block-size': 256, 'lr': 1e-5 if MODE == 'instruct' else 1e-4,
        'seed': 1337, 'eval-interval': 100, 'eval-iters': 5, 'checkpoint-interval': 100, 'warmup': 20}
    if MODE == 'instruct':
        training.update({'steps': steps, 'duplication': 1, 'max-convs': 0, 'grounded-dir': '',
                         'hf-seed-dir': str(PREPARED / 'sft_train'), 'sft-val-dir': str(PREPARED / 'sft_val')})
    else:
        training['run-iters'] = steps
        domain_path = Path(next(iter(m['domains'].values())))
        inputs['data'] = {'path': str(domain_path), 'sha256': sha(domain_path)}
        inputs['mixture'] = {'path': str(PREPARED / 'mixture.json'), 'sha256': sha(PREPARED / 'mixture.json')}
        training['low-vram'] = True
    if resume: training['resume'] = True
    else: inputs['parent'] = {'path': str(parent_path), 'sha256': sha(parent_path)}
    return {'schema_version': 1, 'inputs': inputs, 'training': training, 'output_checkpoint': str(output)}
OUTPUT = ROOT / 'checkpoints' / ('kaggle_v4_' + MODE + '.pt')
SMOKE = ROOT / 'smoke_run.json'
atomic_json(str(SMOKE), recipe(CHECKPOINT_INPUT, OUTPUT, 20))
launcher = [sys.executable, str(ROOT / 'scripts/python_d.py'), 'scripts/run_training.py', '--config', str(SMOKE)]
subprocess.run(launcher, check=True)  # preflight, chưa train
import time as _time
_t0 = _time.time()
subprocess.run(launcher + ['--execute'], check=True)  # 20 optimizer updates, tự save cuối run
_smoke_s = _time.time() - _t0
print(f'Smoke: {_smoke_s:.1f}s / 20 bước (gồm load model+data) -> ~{20 / _smoke_s:.2f} it/s thô; đối chiếu dòng it/s cumulative trong log')
assert OUTPUT.is_file(), 'Smoke không tạo checkpoint'
print('SMOKE PASS:', OUTPUT)


In [ ]:
if RUN_LONG:
    CAP = 100000 if ALLOW_LONG else 5000
    assert 1 <= STEPS <= CAP, f'STEPS {STEPS:,} vượt trần {CAP:,}; đặt ALLOW_LONG=True ở ô 1 nếu bạn chấp nhận số vòng lặp data'
    if globals().get('MODE', 'instruct') == 'instruct':
        EPOCH_STEPS = 8500  # đo trên prepared_v4: 34.563 block / grad-accum 4 (block 256); Kaggle 17.440 hội thoại ~ 8,4-8,7k
        print(f'Estimate: {STEPS:,} steps ~ {STEPS / EPOCH_STEPS:.1f} epochs over the current SFT data; keep _best.pt if val gets worse')
    else:
        TOK_PER_BYTE = 0.241  # đo sample 256KB/ngành trên general_v1 (0,21-0,39 tok/B); chỉ là ước lượng
        tokens_per_step = 1 * 4 * 256  # batch 1 x grad-accum 4 x block 256 (ô 4)
        tokens = sum(Path(p).stat().st_size for p in m['domains'].values()) * TOK_PER_BYTE
        epoch_steps = tokens / tokens_per_step
        n_domains = len(m['domains'])
        print(f'Pretrain: ~{tokens / 1e6:.1f}M token trong {n_domains} ngành, {tokens_per_step} token/bước -> 1 vòng ~ {epoch_steps:,.0f} bước; {STEPS:,} bước ~ {STEPS / epoch_steps:.2f} vòng (ước lượng từ kích thước file x 0,241 tok/B)')
        print(f'Ước lượng {STEPS:,} bước ~ {STEPS / 1.914 / 3600:.1f} giờ ở 1,914 it/s (số đo run v4 cùng recipe); xác nhận bằng it/s thật in ở ô 4.')
    LONG = ROOT / 'continued_run.json'
    atomic_json(str(LONG), recipe(OUTPUT, OUTPUT, STEPS, resume=True))
    cmd = [sys.executable, str(ROOT / 'scripts/python_d.py'), 'scripts/run_training.py', '--config', str(LONG)]
    subprocess.run(cmd, check=True)
    subprocess.run(cmd + ['--execute'], check=True)
else:
    print('Chỉ smoke. Muốn run dài: RUN_LONG=True ở ô 1 rồi Save Version / Save & Run All trong phiên mới.')


In [ ]:
# Không nhân đôi checkpoint GB. Files ở /kaggle/working/jin/checkpoints được lưu như output của version.
files = []
for path in sorted((ROOT / 'checkpoints').glob('*.pt')):
    item = {'path': str(path.relative_to('/kaggle/working')), 'bytes': path.stat().st_size, 'sha256': sha(path)}
    files.append(item); print(item)
atomic_json(str(ROOT / 'output_index.json'), {'mode': MODE, 'files': files, 'trained_steps_requested_after_smoke': STEPS if RUN_LONG else 0})
print('Lưu version/output trước khi hết quota. Tải checkpoint latest + manifest + output_index.json về D, không ghi đè model đang phục vụ.')
